# GeoCalib-Align Colab Notebook

End-to-end notebook for lightweight geoscience fine-tuning, evaluation, and figure generation.

## Section 0: Setup

Install dependencies, verify GPU hardware, and optionally collect a Hugging Face token for gated model downloads. NumPy is pinned below version 2 on purpose to avoid common Colab compatibility issues.

In [ ]:
%%time
!pip install -q torch==2.2.0 transformers==4.40.0 peft==0.10.0 bitsandbytes==0.43.0 datasets==2.19.0 accelerate==0.29.0 trl==0.8.6 bert-score==0.3.13 matplotlib==3.8.4 seaborn==0.13.2 plotly==5.21.0 pandas==2.2.2 "numpy<2" scikit-learn==1.4.2 scipy==1.13.0 tqdm==4.66.2 pyyaml==6.0.1 huggingface_hub==0.23.0 kaleido==0.2.1

In [ ]:
import os
from getpass import getpass
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
gpu_vram = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else 0
print({'gpu_available': torch.cuda.is_available(), 'gpu_name': gpu_name, 'gpu_vram_gb': gpu_vram})
os.environ['HF_TOKEN'] = getpass('Hugging Face token (optional for gated models, leave blank if not needed): ')

## Section 1: Data Loading

Download GeoBench, EarthSE, and GeoSignal automatically, then inspect samples and view token-length statistics.

In [ ]:
%%time
!python data/download_data.py
!python data/prepare_geosignal.py

import json
import matplotlib.pyplot as plt

train_records = json.load(open('data/processed/geosignal_train.json'))
for sample in train_records[:3]:
    print(sample)
lengths = [len((row['instruction'] + ' ' + row['input'] + ' ' + row['output']).split()) for row in train_records]
plt.hist(lengths, bins=30)
plt.title('GeoSignal Token Length Distribution')
plt.xlabel('Approximate word count')
plt.ylabel('Frequency')
plt.show()

## Section 2: Model Loading

Load the base model in 4-bit mode and inspect trainable parameter counts.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config, device_map='auto')
print(model)
print('Trainable parameters before LoRA:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Section 3: Fine-Tuning (Standard LoRA)

In [ ]:
%%time
!python finetune/train_lora_standard.py --model_id meta-llama/Meta-Llama-3.1-8B-Instruct --output_dir results/colab_demo/lora_standard --config config/experiments.yaml

## Section 4: Fine-Tuning (Selective LoRA)

In [ ]:
%%time
!python finetune/train_lora_selective.py --model_id meta-llama/Meta-Llama-3.1-8B-Instruct --output_dir results/colab_demo/lora_selective --config config/experiments.yaml

## Section 5: Fine-Tuning (LoRA + Replay)

In [ ]:
%%time
!python finetune/train_lora_replay.py --model_id meta-llama/Meta-Llama-3.1-8B-Instruct --output_dir results/colab_demo/lora_replay --config config/experiments.yaml

## Section 6: Fine-Tuning (Mix-CPT)

In [ ]:
%%time
!python finetune/train_mix_cpt.py --model_id meta-llama/Meta-Llama-3.1-8B-Instruct --output_dir results/colab_demo/mix_cpt --config config/experiments.yaml

## Section 7: Evaluation

In [ ]:
%%time
!python evaluate/eval_closed_tasks.py --model_path results/colab_demo/lora_standard --model_name Llama-3.1-8B --strategy lora_std
!python evaluate/eval_open_tasks.py --model_path results/colab_demo/lora_standard --model_name Llama-3.1-8B --strategy lora_std --judge_model Qwen/Qwen2.5-3B-Instruct
!python evaluate/aggregate_results.py
import pandas as pd
display(pd.read_csv('results/summary.csv').head())

### PhysGeo Note

`data/build_physgeo_eval.py` now creates an empty annotation template on purpose. Populate `data/physgeo_eval_template.json` with expert-reviewed real examples before running `evaluate/eval_physgeo.py`.

## Section 8: Figure Generation

In [ ]:
%%time
!python figures/generate_all_figures.py --data results/summary.csv

from IPython.display import Image, display
for index in range(1, 9):
    display(Image(filename=f'figures/fig{index}_' + ['tradeoff_scatter','radar_chart','heatmap','bar_strategies','physgeo_breakdown','cost_performance','alignment_delta','leaderboard'][index-1] + '.png'))

## Section 9: Export

In [ ]:
%%time
!zip -r geocalib_align_figures.zip figures/*.pdf figures/*.png results/summary.csv
from google.colab import files
files.download('results/summary.csv')
files.download('geocalib_align_figures.zip')